# Phase 3a — Train Plain Baselines on Colab (T4)

Trains the two **unmodified** baselines from the project spec section 3 on the SOURCE split
(India + Japan) and evaluates them in-domain:

- **Faster R-CNN ResNet-50 FPN** (two-stage) — the *reference baseline*, reproducing
  DA-RDD's detector backbone without its adaptation machinery.
- **YOLOv8n** (one-stage) — the *comparison baseline*, the paradigm that won CRDDC-2022.

> **Framing (this is a viva question).** YOLOv8 is **not** "more advanced" than Faster
> R-CNN. One-stage detectors are faster; two-stage detectors give better localisation and
> recognition accuracy. They are different paradigms, **not** a linear upgrade path — never
> present one as an upgrade of the other (the project spec sections 3 and 12).

No attention module — that is Phase 3b, and it must not be merged into this run
(the project spec section 12: do not build 3b before 3a numbers have been reviewed).

YOLOv8 is trained **first** below purely for scheduling reasons: it finishes in roughly
half the time, so a session drop still leaves you with one complete baseline. The order
carries no claim about model quality.

### Why this notebook exists
The development machine has an AMD GPU, so PyTorch there runs on CPU only. Measured
locally, one YOLOv8n epoch over the full SOURCE split takes **~89 minutes**, and Faster
R-CNN exceeds **10 hours per epoch** — together roughly **8–11 days**. On a T4 the same
work takes a few hours.

### Before you start
1. Run `python scripts/make_colab_bundle.py` locally.
2. Upload `dist/rdd_bundle.zip` to your Google Drive.
3. Set `Runtime -> Change runtime type -> GPU` **before** running any cell.

### Training is CHUNKED — you do not need one long session

Free Colab caps session length, so training is split into chunks that each stop cleanly
and save. **Every chunk cell is the same command**, and rerunning it continues from where
the last one stopped:

- **YOLOv8:** 100 epochs as 5 chunks of 20
- **Faster R-CNN:** 12 epochs as 4 chunks of 3 (its epochs are far slower)

Each chunk cell saves to Drive automatically when it finishes. If a session dies
mid-chunk, you lose at most that chunk — rerun the same cell and it picks up from the
last completed epoch.

This is a **true resume**, not a warm restart: optimizer state, EMA, and the LR schedule
position are all restored, so 5 chunks of 20 produce the same model as one 100-epoch run.
The LR schedule still spans the full 100 epochs rather than restarting five times.

Evaluation and logging happen **only after the final chunk** of each model — a
partially-trained model is never written to `experiment_log.csv` as a result.

## Step 1 — Confirm a GPU is actually attached

This cell **fails deliberately** if there is no CUDA device, rather than silently falling back to CPU and taking days.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "No CUDA GPU. Set Runtime -> Change runtime type -> GPU, then rerun. "
    "Training on CPU takes days (that is the whole reason for this notebook)."
)
print("device:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## Step 2 — Mount Google Drive

When the popup appears: **Connect to Google Drive** → choose your account → tick **all**
permission boxes → **Continue**. Dismissing it early is the usual cause of failure.

### If you get `MessageError: credential propagation was unsuccessful`

That is a **browser** problem, not an account or notebook problem. Colab runs the OAuth
handshake inside an iframe, so anything blocking third-party cookies stops the credential
coming back. In order of likelihood:

1. **Allow third-party cookies.** Chrome → Settings → *Privacy and security* →
   *Third-party cookies* → allow, or add site exceptions for `[*.]google.com` and
   `[*.]colab.research.google.com`. Then **Runtime → Restart session** and rerun.
2. **Turn off ad blockers / privacy extensions** for this tab (uBlock, Privacy Badger,
   Brave Shields). Use a normal window, not Incognito.
3. **Be signed into one Google account only** — with several, the mount can bind to the
   wrong one.
4. **Use the UI instead:** folder icon in the left sidebar → **Mount Drive** button.
5. Still stuck? Skip to **Step 2b** and upload the zip straight into the session — no
   Drive, no OAuth.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Step 2b — FALLBACK ONLY: run without Drive

Skip this cell if Step 2 worked.

This uploads the bundle directly into the Colab VM, avoiding OAuth entirely. Two real
costs, so prefer fixing Step 2 if you can:

- Browser upload of a ~1.3 GB file is slower and flakier than a Drive transfer.
- **The VM's disk is wiped when the session ends.** With no Drive to save to, you must
  download results yourself before disconnecting (the last cell handles this), or the
  training is lost.

In [ ]:
# FALLBACK ONLY -- skip if Step 2 (Drive mount) succeeded.
from google.colab import files

uploaded = files.upload()          # select rdd_bundle.zip in the file picker
BUNDLE_ZIP = '/content/' + next(iter(uploaded))
print("uploaded to:", BUNDLE_ZIP)

## Step 3 — Unpack the bundle

Edit `BUNDLE_ZIP` to wherever you put the zip in your Drive.

In [ ]:
import zipfile, os, time
from pathlib import Path

# If Step 2b already set BUNDLE_ZIP (the no-Drive fallback), keep that value.
# Otherwise default to Drive. EDIT the path if your zip is not at My Drive root.
try:
    BUNDLE_ZIP
except NameError:
    BUNDLE_ZIP = '/content/drive/MyDrive/rdd_bundle.zip'
print("using bundle:", BUNDLE_ZIP)

REPO = Path('/content/road-damage-detection')

assert Path(BUNDLE_ZIP).exists(), f"Not found: {BUNDLE_ZIP} -- check the path in your Drive"

# Test for a FILE that only a complete extract produces, not just the directory.
# The directory can exist while being empty or half-populated (an interrupted
# extract, or a dir created by some other step), and skipping extraction then
# fails much later with a confusing "can't open file src/models/train_yolo.py".
SENTINEL = REPO / 'src' / 'models' / 'train_yolo.py'

if SENTINEL.exists():
    print("code already extracted, skipping unpack")
else:
    REPO.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(BUNDLE_ZIP) as zf:
        zf.extractall(REPO)
    print(f"Extracted in {time.time() - t0:.0f}s")

assert SENTINEL.exists(), (
    f"{SENTINEL} still missing after extract -- the zip is incomplete or is an "
    f"older bundle without the chunked training scripts. Re-upload rdd_bundle.zip."
)

os.chdir(REPO)
print("cwd:", os.getcwd())
print(sorted(os.listdir(REPO)))

## Step 4 — Install dependencies

Colab already ships torch/torchvision with CUDA, so only ultralytics is needed.

In [ ]:
!pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)

## Step 5 — Point the dataset configs at Colab paths

The YAMLs are generated with absolute paths (ultralytics resolves relative dataset
paths against its own settings dir, not the repo — a classic silent "0 images found"
failure). They ship with Windows paths, so regenerate them here.

> **Order matters: always run this AFTER Step 3.** Extracting the bundle restores the
> Windows-path YAMLs, so regenerating first and then extracting silently undoes this.
> Symptom if you get it wrong: ultralytics fails with a concatenated path like
> `/content/road-damage-detection/config/D:/road-damage-detection/...`. Step 6 now
> asserts these paths resolve, so it will be caught before any training runs.

In [ ]:
!python src/data/write_dataset_configs.py --data-root /content/road-damage-detection/data/processed
print()
!cat config/dataset_source.yaml

## Step 6 — Verify the data arrived intact, and run the unit tests

Counts here must match `data/split_report.json` (12,748 train / 2,732 val / 2,732 test).
If they do not, the upload was incomplete — stop and re-upload rather than training on
partial data.

In [ ]:
from pathlib import Path

for split in ['train', 'val', 'test']:
    imgs = len(list(Path(f'data/processed/source/images/{split}').glob('*.jpg')))
    lbls = len(list(Path(f'data/processed/source/labels/{split}').glob('*.txt')))
    print(f"{split:5s}: {imgs:6d} images  {lbls:6d} labels")

expected = {'train': 12748, 'val': 2732, 'test': 2732}
for split, n in expected.items():
    got = len(list(Path(f'data/processed/source/images/{split}').glob('*.jpg')))
    assert got == n, f"{split}: expected {n} images, found {got} -- upload incomplete"
print("Counts match split_report.json.")

# The dataset YAMLs ship with Windows paths and are restored verbatim by every
# extract. If Step 5 was skipped -- or was run BEFORE an extract and then
# clobbered -- ultralytics fails much later with a confusing concatenated path
# like '/content/road-damage-detection/config/D:/road-damage-detection/...'.
# Catch it here, before committing to a training chunk.
import yaml
cfg = yaml.safe_load(open('config/dataset_source.yaml'))
for key in ('train', 'val', 'test'):
    p = Path(cfg[key])
    assert p.is_absolute() and p.exists(), (
        f"config/dataset_source.yaml '{key}' points at {cfg[key]} which does not exist "
        f"on this machine.\nRe-run Step 5 (write_dataset_configs.py) -- and always run it "
        f"AFTER extracting, since the extract restores the Windows-path YAMLs."
    )
print("Dataset config paths resolve correctly.")

In [ ]:
# Known-answer tests: metric correctness and the YOLO -> torchvision conversion.
!python src/eval/test_metrics.py
print()
!python src/models/test_rcnn_dataset.py

## Step 6b — Define helpers AND restore previous progress

> ### Run this cell in EVERY session, before any training chunk.
> It does two things that are easy to miss:
>
> 1. **Defines `save_results()`**, which every chunk cell calls. Function definitions live
>    in kernel memory, so a restart or reconnect loses them — that is the
>    `NameError: name 'save_results' is not defined` you get if you skip this.
> 2. **Restores training progress from Drive.** Step 3 unpacks *code and data* only; it
>    does not bring back checkpoints. If the VM was recycled, `experiments/` is empty, and
>    without this restore the next chunk would find no run directory and **silently start
>    again from epoch 1**, throwing away every completed chunk.
>
> It is safe to run repeatedly — restoring simply overwrites the local copy with whatever
> is in Drive, which is the newer of the two by construction.

In [ ]:
import shutil, os
from pathlib import Path

def restore_results():
    """Bring back checkpoints/logs saved by earlier chunks, possibly in another session.

    Without this, a recycled VM leaves experiments/ empty and training restarts from
    scratch instead of resuming -- a silent loss of every completed chunk.
    """
    src_root = Path('/content/drive/MyDrive/rdd_results')
    if not src_root.exists():
        print("No saved results in Drive yet -- this is a fresh start.")
        return
    for name in ['runs', 'results']:
        src = src_root / name
        if src.exists():
            dst = Path('experiments') / name
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print("restored", dst)

def save_results():
    """Copy results to Drive, or fall back to a browser download if Drive is absent."""
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        out = drive_root / 'rdd_results'
        out.mkdir(parents=True, exist_ok=True)
        for src in [Path('experiments/runs'), Path('experiments/results')]:
            if src.exists():
                dst = out / src.name
                if dst.exists():
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
        print("saved to Drive:", out)
        return out
    print("Drive not mounted -- packing results for download instead.")
    archive = shutil.make_archive('/content/rdd_results', 'zip', 'experiments')
    print("archive:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
    from google.colab import files
    files.download(archive)
    return Path(archive)

def progress():
    """Show how many epochs each run has completed so far."""
    import csv as _csv, yaml as _yaml
    for run in sorted(Path('experiments/runs').glob('*')):
        rc, ay = run / 'results.csv', run / 'args.yaml'
        if rc.exists():
            done = max(0, sum(1 for _ in _csv.reader(open(rc))) - 1)
            total = (_yaml.safe_load(open(ay)) or {}).get('epochs', '?') if ay.exists() else '?'
            print(f"  {run.name}: {done}/{total} epochs")
        elif (run / 'last.pt').exists():
            import torch
            ck = torch.load(run / 'last.pt', map_location='cpu', weights_only=False)
            print(f"  {run.name}: {ck['epoch']}/{ck['config']['epochs']} epochs")

restore_results()
print("\nhelpers ready. current progress:")
progress()

## Step 7 — (Recommended) Smoke test on the subset first

Two minutes on a T4. Catches path/config problems before committing to the full run —
this is exactly how the Windows path bug in the evaluation code was caught locally.

Numbers from this cell are **smoke-test artifacts, not results**. They are logged with
phase `3a-smoke` and must never appear in the report.

In [ ]:
!python src/models/train_yolo.py \
    --data config/dataset_source_subset.yaml \
    --name yolov8n_smoke --epochs 2 --smoke --device 0

## Step 8 — Train YOLOv8n: 100 epochs as 5 chunks of 20

Hyperparameters come from `config/hyperparams.yaml` (100 epochs total, 640px, batch 16,
patience 20, seed 42). Roughly **30–45 min per chunk** on a T4.

> **Returning to a new session?** Re-run **Steps 1–6b** first (GPU, mount, unpack,
> install, configs, helpers+restore). Those set the working directory, define
> `save_results()`, and pull your checkpoints back from Drive. Jumping straight to a chunk
> cell in a fresh kernel gives `can't open file 'src/models/train_yolo.py'` and
> `NameError: save_results is not defined` — and if the VM was recycled, would restart
> training from epoch 1.

**All five cells below run the identical command.** The script reads how many epochs are
already done and continues from there, so:

- Run them in order, one per sitting if you like.
- If a session dies mid-chunk, just rerun that same cell.
- If you accidentally run a cell twice after it completed, it prints
  *"Training already complete"* and skips — it will not retrain or corrupt anything.

Each cell saves to Drive on completion. Evaluation and the `experiment_log.csv` row
happen automatically at the end of **chunk 5**, not before.

### YOLOv8 — chunk 1 of 5 (epochs 1–20)

In [ ]:
!python src/models/train_yolo.py \
    --data config/dataset_source.yaml \
    --name yolov8n_source \
    --chunk-epochs 20 \
    --device 0

save_results()   # chunk 1/5 done -- persist before anything can drop

### YOLOv8 — chunk 2 of 5 (epochs 21–40)

In [ ]:
!python src/models/train_yolo.py \
    --data config/dataset_source.yaml \
    --name yolov8n_source \
    --chunk-epochs 20 \
    --device 0

save_results()   # chunk 2/5 done -- persist before anything can drop

### YOLOv8 — chunk 3 of 5 (epochs 41–60)

In [ ]:
!python src/models/train_yolo.py \
    --data config/dataset_source.yaml \
    --name yolov8n_source \
    --chunk-epochs 20 \
    --device 0

save_results()   # chunk 3/5 done -- persist before anything can drop

### YOLOv8 — chunk 4 of 5 (epochs 61–80)

In [ ]:
!python src/models/train_yolo.py \
    --data config/dataset_source.yaml \
    --name yolov8n_source \
    --chunk-epochs 20 \
    --device 0

save_results()   # chunk 4/5 done -- persist before anything can drop

### YOLOv8 — chunk 5 of 5 (epochs 81–100)

In [ ]:
!python src/models/train_yolo.py \
    --data config/dataset_source.yaml \
    --name yolov8n_source \
    --chunk-epochs 20 \
    --device 0

save_results()   # chunk 5/5 done -- persist before anything can drop

## Step 9 — Train Faster R-CNN: 12 epochs as 4 chunks of 3

Faster R-CNN epochs are far more expensive than YOLO's, so the chunks are 3 epochs each —
roughly **1–1.5 hours per chunk** on a T4.

Same rules as above: identical command in every cell, rerun to resume, safe to re-run
after completion. Resume restores the optimizer *and* the LR scheduler, so the step decay
at epoch 8 lands correctly regardless of where the chunk boundaries fell.

`--val-max-images` caps the per-epoch validation check to keep chunks shorter. The final
test-split evaluation always uses all 2,732 images regardless of this setting.

### Faster R-CNN — chunk 1 of 4 (epochs 1–3)

In [ ]:
!python src/models/train_faster_rcnn.py \
    --data-root data/processed/source \
    --name faster_rcnn_source \
    --chunk-epochs 3 \
    --val-max-images 800 \
    --device cuda

save_results()   # chunk 1/4 done -- persist before anything can drop

### Faster R-CNN — chunk 2 of 4 (epochs 4–6)

In [ ]:
!python src/models/train_faster_rcnn.py \
    --data-root data/processed/source \
    --name faster_rcnn_source \
    --chunk-epochs 3 \
    --val-max-images 800 \
    --device cuda

save_results()   # chunk 2/4 done -- persist before anything can drop

### Faster R-CNN — chunk 3 of 4 (epochs 7–9)

In [ ]:
!python src/models/train_faster_rcnn.py \
    --data-root data/processed/source \
    --name faster_rcnn_source \
    --chunk-epochs 3 \
    --val-max-images 800 \
    --device cuda

save_results()   # chunk 3/4 done -- persist before anything can drop

### Faster R-CNN — chunk 4 of 4 (epochs 10–12)

In [ ]:
!python src/models/train_faster_rcnn.py \
    --data-root data/processed/source \
    --name faster_rcnn_source \
    --chunk-epochs 3 \
    --val-max-images 800 \
    --device cuda

save_results()   # chunk 4/4 done -- persist before anything can drop

### Progress check (run any time)

In [ ]:
progress()

## Step 10 — Benchmark speed and parameter count for both

the project spec section 7 (Phase 3a) requires benchmarking inference speed **and** parameter
count for both models, and section 5.3 requires them measured on **identical hardware**.
Running both in this one cell guarantees that.

Latency is measured at batch size 1 (per-image response time, which is what a deployed
road survey cares about). A large gap between mean and median means the measurement was
disturbed by something else on the machine — rerun it if you see one.

In [ ]:
!python src/eval/benchmark_speed.py \
    --yolo-weights experiments/runs/yolov8n_source/weights/best.pt \
    --rcnn-weights experiments/runs/faster_rcnn_source/best.pt \
    --images data/processed/source/images/test \
    --device cuda

## Step 11 — Review the in-domain results

These are the Phase 3a numbers: the reference point every later phase is measured against.

In [ ]:
import pandas as pd

log = pd.read_csv('experiments/results/experiment_log.csv')
# Exclude smoke-test rows -- they are pipeline artifacts, not results.
real = log[~log['phase'].astype(str).str.contains('smoke')]

cols = ['run_id', 'model', 'dataset', 'split', 'num_images', 'map50', 'map50_95',
        'ap_D00', 'ap_D10', 'ap_D20', 'ap_D40', 'precision', 'recall', 'f1',
        'latency_ms', 'fps', 'param_count']
print(real[cols].to_string(index=False))

if real.empty:
    print("No non-smoke rows yet -- training has not produced reportable results.")

## Final step — Save everything back to Drive

Then download `experiments/results/experiment_log.csv` and copy it into the local repo —
the log is version-controlled even though weights are not.

In [ ]:
out = save_results()

if out.is_dir():
    print("\nContents:")
    for p in sorted(out.rglob('*')):
        if p.is_file():
            print(" ", p.relative_to(out), f"{p.stat().st_size / 1e6:.1f} MB")

print("\nPhase 3a complete. Stop here for student review before Phase 3b (the project spec section 3).")